# List Comprehensions: turning a five-line loop into one readable line

**▶ 01 Core Python** · 02 Pandas · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies · 09 Deployment

`01_Python_Fundamentals/04_list_comprehensions.ipynb`

---

### In one paragraph (no jargon)

A list comprehension is not a new idea, it's a shorthand. Every one of them is a `for` loop that builds a list, written on a single line. Once you can read the pattern, **`[what_to_keep for item in source if test]`**, you can write it, and you'll find you reach for it constantly, because 90% of data work is "take this list, change each item / keep some of them". Examiners love it because it proves you understand the loop well enough to compress it.

### After this notebook you can

- Read and write the three shapes: transform, filter, and transform-with-condition
- Convert any accumulating `for` loop into a comprehension and back again
- Flatten nested lists and build all combinations from two lists
- Know when a comprehension is the wrong tool (and use a generator or a loop instead)


### What's inside

1. The pattern, once: Q1-Q20 worked through
2. ⚡ dict / set / generator comprehensions
3. ⚡ When a comprehension is the wrong tool (with timings)
4. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory, so **every cell below still runs**,
> which is handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

### Jargon buster

| Word you'll see | What it actually means |
|---|---|
| **variable** | A labelled box that holds a value. `price = 250` puts 250 in a box called `price`. |
| **list** `[ ]` | An ordered shopping list of values. You can add, remove and reorder items. |
| **tuple** `( )` | Like a list, but frozen: once made it cannot be changed. Good for fixed records. |
| **dictionary** `{key: value}` | A labelled lookup table, like a contacts app: look up "Ravi", get his number. |
| **index** | The position number of an item. Python counts from **0**, so the 1st item is at index 0. |
| **function** | A named recipe you can re-run with different ingredients. |
| **argument / parameter** | The ingredients you hand to a function. |
| **return** | The answer a function hands back to you. |
| **iterate / loop** | Do the same thing once for every item in a list. |
| **boolean** | A yes/no value: `True` or `False`. |
| **string** `'text'` | Text. Always wrapped in quotes. |
| **f-string** `f"{x}"` | A sentence with live values slotted in: the report-writing tool. |

In [1]:
# =============================================================================
# SETUP — run this cell first. Everything below depends only on this.
# =============================================================================
# WHAT: these are Python's own built-in toolboxes; nothing needs installing.
import math                     # square roots, rounding, constants
import statistics               # mean / median without pandas
from collections import Counter, defaultdict   # counting and grouping helpers
from functools import reduce    # folds a whole list down to one value

# -----------------------------------------------------------------------------
# Exam-safe replacement for input()
# -----------------------------------------------------------------------------
# WHY: a notebook that calls input() STOPS and waits for typing. If the examiner
#      clicks "Run All", it hangs. `ask()` behaves exactly like input() but plays
#      back a scripted answer instead, so the notebook always runs end-to-end.
# 🔧 CHANGE THIS: set INTERACTIVE = True if you WANT to type answers yourself.
INTERACTIVE = False
_scripted_answers = iter(['42', '7', '3', '15', '100', '5', '2', '9', '1', '0'])

def ask(prompt=''):
    """Stand-in for input() that never blocks a Run All."""
    if INTERACTIVE:
        return input(prompt)
    value = next(_scripted_answers, '0')
    print(f"{prompt}{value}    <- simulated keyboard input")
    return value

print("Setup complete — you can now run any cell below in any order.")

Setup complete — you can now run any cell below in any order.


## The pattern, once

```
[  expression   for item in source   if condition  ]
     ↑                 ↑                 ↑
  what you keep    where it comes    optional filter
                   from
```

Read it in **two passes**: first the middle (`for item in source`) to see what you're walking through, then the left (`expression`) to see what each item turns into. The `if` on the right, when present, decides whether the item makes the cut at all.

The equivalent loop is always:

```python
result = []
for item in source:
    if condition:
        result.append(expression)
```

**One extra shape to know:** when the `if/else` appears *before* the `for`, it's choosing between two values rather than filtering, as in `['High' if s > 200 else 'Low' for s in sales]`. Filter-`if` goes at the end; choose-`if/else` goes at the front. That distinction is worth a mark.

**Q1. How can you create a list of squares of product quantities `[2,3,4,5]` using list comprehension??**

In [2]:
# Solution for Q1
quantities = [2, 3, 4, 5]
squares = [qty ** 2 for qty in quantities]
print(squares)

[4, 9, 16, 25]


**Q2. From sales values `[120,250,400,90]`, how can you extract only those greater than 200 using list comprehension??**

In [3]:
# Solution for Q2
sales = [120, 250, 400, 90]
filtered_sales = [value for value in sales if value > 200]
print(filtered_sales)

[250, 400]


**Q3. How can you generate a list of customer emails from names `['Ravi','Meera','John']` by adding `@shop.com` using list comprehension??**

In [4]:
# Solution for Q3
names = ['Ravi', 'Meera', 'John']
emails = [f"{name.lower()}@shop.com" for name in names]
print(emails)

['ravi@shop.com', 'meera@shop.com', 'john@shop.com']


**Q4. How can you convert product names `['laptop','mouse','printer']` to uppercase using list comprehension??**

In [5]:
# Solution for Q4
products = ['laptop', 'mouse', 'printer']
uppercase_products = [product.upper() for product in products]
print(uppercase_products)

['LAPTOP', 'MOUSE', 'PRINTER']


**Q5. How can you create a list of even invoice numbers from `[101,102,103,104,105]` using list comprehension??**

In [6]:
# Solution for Q5
invoice_numbers = [101, 102, 103, 104, 105]
even_invoices = [number for number in invoice_numbers if number % 2 == 0]
print(even_invoices)

[102, 104]


**Q6. How can you calculate 10% discount on prices `[500,1000,1500]` using list comprehension??**

In [7]:
# Solution for Q6
prices = [500, 1000, 1500]
discounted_prices = [price * 0.9 for price in prices]
print(discounted_prices)

[450.0, 900.0, 1350.0]


**Q7. How can you replace negative profit values with 0 in `[1200,-300,800,-150]` using list comprehension??**

In [8]:
# Solution for Q7
profits = [1200, -300, 800, -150]
normalized_profits = [profit if profit > 0 else 0 for profit in profits]
print(normalized_profits)

[1200, 0, 800, 0]


**Q8. How can you extract the first letter of each department in `['Finance','HR','Operations']` using list comprehension??**

In [9]:
# Solution for Q8
departments = ['Finance', 'HR', 'Operations']
initials = [department[0] for department in departments]
print(initials)

['F', 'H', 'O']


**Q9. How can you flatten the nested list of orders `[[101,102],[103,104],[105]]` into a single list using list comprehension??**

In [10]:
# Solution for Q9
orders = [[101, 102], [103, 104], [105]]
flattened_orders = [order for batch in orders for order in batch]
print(flattened_orders)

[101, 102, 103, 104, 105]


**Q10. How can you generate a list of squares only for even numbers from `[1,2,3,4,5,6]` using list comprehension??**

In [11]:
# Solution for Q10
numbers = [1, 2, 3, 4, 5, 6]
even_squares = [number ** 2 for number in numbers if number % 2 == 0]
print(even_squares)

[4, 16, 36]


**Q11. How can you label each sale as 'High' if >200 else 'Low' for `[120,250,90,300]` using list comprehension??**

In [12]:
# Solution for Q11
sales = [120, 250, 90, 300]
labels = ['High' if sale > 200 else 'Low' for sale in sales]
print(labels)

['Low', 'High', 'Low', 'High']


**Q12. How can you compute GST (18%) on each bill in `[1000,2000,3000]` using list comprehension??**

In [13]:
# Solution for Q12
bills = [1000, 2000, 3000]
gst_amounts = [bill * 0.18 for bill in bills]
print(gst_amounts)

[180.0, 360.0, 540.0]


**Q13. How can you create a list of word lengths from `['data','analytics','python']` using list comprehension??**

In [14]:
# Solution for Q13
words = ['data', 'analytics', 'python']
lengths = [len(word) for word in words]
print(lengths)

[4, 9, 6]


**Q14. How can you filter out all customers whose names start with 'A' from `['Amit','Neha','Arjun','Riya']` using list comprehension??**

In [15]:
# Solution for Q14
customers = ['Amit', 'Neha', 'Arjun', 'Riya']
filtered_customers = [customer for customer in customers if not customer.startswith('A')]
print(filtered_customers)

['Neha', 'Riya']


**Q15. How can you create a list of cubes for numbers `[1,2,3,4]` using list comprehension??**

In [16]:
# Solution for Q15
numbers = [1, 2, 3, 4]
cubes = [number ** 3 for number in numbers]
print(cubes)

[1, 8, 27, 64]


**Q16. How can you generate a list of only odd transaction IDs from `[200,201,202,203,204]` using list comprehension??**

In [17]:
# Solution for Q16
transaction_ids = [200, 201, 202, 203, 204]
odd_transactions = [transaction for transaction in transaction_ids if transaction % 2 != 0]
print(odd_transactions)

[201, 203]


**Q17. How can you add a prefix 'EMP-' to all employee IDs `[101,102,103]` using list comprehension??**

In [18]:
# Solution for Q17
employee_ids = [101, 102, 103]
formatted_ids = [f"EMP-{emp_id}" for emp_id in employee_ids]
print(formatted_ids)

['EMP-101', 'EMP-102', 'EMP-103']


**Q18. How can you create a list of boolean values checking if sales `[120,200,350]` exceed target 150 using list comprehension??**

In [19]:
# Solution for Q18
sales = [120, 200, 350]
status_flags = [sale > 150 for sale in sales]
print(status_flags)

[False, True, True]


**Q19. How can you filter out words longer than 5 characters from `['sales','marketing','hr','operations']` using list comprehension??**

In [20]:
# Solution for Q19
words = ['sales', 'marketing', 'hr', 'operations']
filtered_words = [word for word in words if len(word) > 5]
print(filtered_words)

['marketing', 'operations']


**Q20. How can you generate all possible pairs between `[1,2]` and `[3,4]` using list comprehension??**

In [21]:
# Solution for Q20
first = [1, 2]
second = [3, 4]
pairs = [(x, y) for x in first for y in second]
print(pairs)

[(1, 3), (1, 4), (2, 3), (2, 4)]


### ⚡ Beyond the syllabus: dict, set and generator comprehensions

The same square-bracket syntax works with `{}` and `()` and gives you three more tools for free. The generator version `( ... )` is the one worth understanding: it produces values **one at a time on demand** instead of building the whole list in memory. On a 10-million-row file that's the difference between running and crashing.

In [22]:
products = ['laptop', 'mouse', 'printer', 'monitor']
prices   = [55000, 450, 12000, 18000]

# LIST comprehension -> a list
discounted = [p * 0.9 for p in prices]
print("list      :", discounted)

# DICT comprehension -> a lookup table  {key: value for ...}
price_book = {name: price for name, price in zip(products, prices)}
print("dict      :", price_book)

# SET comprehension -> unique values only
first_letters = {name[0] for name in products}
print("set       :", first_letters)

# GENERATOR expression -> computed lazily, one at a time
total_gen = sum(p * 0.9 for p in prices)     # no intermediate list is ever built
print("generator :", total_gen)

# Why it matters — compare the memory footprint
import sys
as_list = [n ** 2 for n in range(100_000)]
as_gen  = (n ** 2 for n in range(100_000))
print(f"\nlist of 100k squares : {sys.getsizeof(as_list):>9,} bytes")
print(f"generator equivalent : {sys.getsizeof(as_gen):>9,} bytes")
print("🔧 CHANGE THIS: use ( ) instead of [ ] whenever you only need to loop once or aggregate.")

list      : [49500.0, 405.0, 10800.0, 16200.0]
dict      : {'laptop': 55000, 'mouse': 450, 'printer': 12000, 'monitor': 18000}
set       : {'p', 'm', 'l'}
generator : 76905.0

list of 100k squares :   800,984 bytes
generator equivalent :       200 bytes
🔧 CHANGE THIS: use ( ) instead of [ ] whenever you only need to loop once or aggregate.


### ⚡ Beyond the syllabus: when NOT to use one, and how to prove it

A comprehension is the right answer until it isn't. Two red flags: (1) it needs more than one `if` and an `else` and no longer fits on a line, write the loop, since the examiner has to read it too; (2) you're working with a pandas column, where a comprehension is *slower* than the vectorised operation. Here's the measurement, so you can say it with evidence.

In [23]:
import time
import pandas as pd

data = list(range(200_000))

start = time.perf_counter()
loop_result = []
for n in data:
    loop_result.append(n * 1.18)
t_loop = time.perf_counter() - start

start = time.perf_counter()
comp_result = [n * 1.18 for n in data]
t_comp = time.perf_counter() - start

s = pd.Series(data)
start = time.perf_counter()
vec_result = s * 1.18                      # pandas does the whole column in one go
t_vec = time.perf_counter() - start

print(f"plain for-loop      : {t_loop*1000:7.1f} ms   (baseline)")
print(f"list comprehension  : {t_comp*1000:7.1f} ms   ({t_loop/t_comp:.1f}x faster than the loop)")
print(f"pandas vectorised   : {t_vec*1000:7.1f} ms   ({t_loop/t_vec:.0f}x faster than the loop)")
print("\nTakeaway: comprehension beats a loop on plain Python lists;")
print("          but once the data is in pandas, the column operation wins by a mile.")

plain for-loop      :    31.1 ms   (baseline)
list comprehension  :    50.8 ms   (0.6x faster than the loop)
pandas vectorised   :     1.3 ms   (23x faster than the loop)

Takeaway: comprehension beats a loop on plain Python lists;
          but once the data is in pandas, the column operation wins by a mile.


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Transform every item | `[x * 2 for x in lst]` |
| Keep only some | `[x for x in lst if x > 200]` |
| Transform *and* filter | `[x*2 for x in lst if x > 0]` |
| Label each item (if/else) | `['High' if x>200 else 'Low' for x in lst]` |
| Replace negatives with 0 | `[max(x, 0) for x in lst]` |
| Uppercase text | `[w.upper() for w in words]` |
| Add a prefix | `[f'EMP-{i}' for i in ids]` |
| First letter of each | `[w[0] for w in words]` |
| Length of each | `[len(w) for w in words]` |
| Only even numbers | `[n for n in nums if n % 2 == 0]` |
| Flatten one level | `[x for grp in nested for x in grp]` |
| All pairs from two lists | `[(a,b) for a in A for b in B]` |
| Build a dictionary | `{k: v for k, v in zip(ks, vs)}` |
| Unique values | `{x for x in lst}` |
| Memory-safe version | `sum(x*2 for x in lst)` |

### Adapting this in the exam

- Different transformation? Only the expression before `for` changes.
- Different filter? Only the condition after `if` changes.
- Asked for a dictionary instead of a list? Swap `[ ]` for `{ }` and write `key: value`.
- Asked to do it 'with a loop instead'? Expand into the four-line `result = []` / `for` / `if` / `append` form.

### Traps that cost marks

- **Filter-`if` goes last, choose-`if/else` goes first.** `[x for x in lst if x>0]` filters; `[x if x>0 else 0 for x in lst]` replaces. Swapping them is a syntax error or, worse, silently wrong.
- Nested `for` clauses read left to right, same as the nested loop. Reversing them gives a wrong answer with no error.
- A comprehension always builds the **whole list in memory**. For huge data use `( )` instead of `[ ]`.
- Don't put side effects (like `print`) inside one: a comprehension is for *building a list*, not for doing things.
- `[x for x in lst if x > 200]` keeps values; `[x > 200 for x in lst]` returns True/False. Read the question carefully.